<a href="https://colab.research.google.com/github/Sumit-Pathrabe/Ai_Model_for_Tech_and_Finance/blob/main/Ai_Model_for_Tech_and_Finance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install Unsloth and specific library versions to prevent version conflicts
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7yjx4gq8/unsloth_f84e20f56547410197792da5f444eb74
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7yjx4gq8/unsloth_f84e20f56547410197792da5f444eb74
  Resolved https://github.com/unslothai/unsloth.git to commit af6504f900fe611a056e66eec6ab74976eab7f34
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # Plenty of space for document summaries
dtype = None          # Auto-detects Colab's GPU setup
load_in_4bit = True   # Keeps the model lightweight and fast

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct", # Corrected model name
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    trust_remote_code = True,
)

# Apply PEFT/LoRA to only train 1% of the model's weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Model loaded and optimized successfully!")

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model loaded and optimized successfully!


In [6]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Structuring your exact requirements: Tech/Finance jargon, professional answers, and summaries
data = [
    # Task 1: Understand Technical Jargon
    {"instruction": "Define what a Kubernetes Pod is and explain its relation to Docker containers.",
     "output": "A Pod is the smallest deployable unit in Kubernetes, representing a single instance of a running process. A Pod wraps one or more Docker containers, sharing networking, storage, and a unique IP address space, allowing tightly coupled containers to scale together efficiently."},

    # Task 2: Understand Financial Jargon
    {"instruction": "What is Equity Dilution and how does it happen in startups?",
     "output": "Equity dilution occurs when a company issues new shares of stock, reducing the ownership percentage of existing shareholders. This commonly occurs during venture capital funding rounds, stock option exercises for employees, or secondary public offerings."},

    # Task 3: Answer Industry Questions Accurately
    {"instruction": "Can you calculate the P/E Ratio of a company with a stock price of $150 and EPS of $5? Explain what this implies professionally.",
     "output": "The Price-to-Earnings (P/E) Ratio is calculated as Stock Price / Earnings Per Share (EPS). Here, $150 / $5 = 30. Professionally, a P/E of 30 means investors are willing to pay $30 for every $1 of current earnings, usually signaling high expectations for future enterprise growth or industry-specific tech scaling."},

    # Task 4: Summarize Technical Documents
    {"instruction": "Summarize the following release notes document:\n'We have migrated our REST APIs to native Microservices managed via Kubernetes. All deployment strategies now execute automatically via our new CI/CD pipeline, lowering infrastructure costs via optimized Docker container configurations.'",
     "output": "Summary: The technical infrastructure has completed a modernization migration moving legacy REST APIs into a Kubernetes-managed microservices architecture. Software releases are fully automated using a CI/CD pipeline, resulting in lowered infrastructure overhead thanks to optimized Docker resource utilization."},

    # Task 5: Summarize Financial Documents & Generate Professional Responses
    {"instruction": "Summarize this quarterly financial report excerpt:\n'Our EBITDA surged by 15% due to aggressive scale in our tech vertical, offsetting a brief dip in traditional asset growth. The balance sheet analysis indicates strong cash retention, paving the way for systematic SIP investments next fiscal year.'",
     "output": "Summary: The company reported a 15% growth in EBITDA driven primarily by its technology vertical, which countered slower traditional business segments. Due to a healthy balance sheet and robust liquid cash retention, management plans to initiate capital deployments via Systematic Investment Plans (SIP) next fiscal year."}
]

# Apply the formal Llama-3 structural prompt format
tokenizer = get_chat_template(tokenizer, chat_template = "llama-3")

def format_prompts(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{output}<|eot_id|>"
        texts.append(text)
    return { "text" : texts }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompts, batched = True)
print("Dataset compiled and mapped.")


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset compiled and mapped.


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2,
        warmup_steps = 2,
        max_steps = 20, # Short training path tailored to our dataset size
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

# Start fine-tuning
trainer_stats = trainer.train()

num_proc must be <= 5. Reducing num_proc to 5 for dataset of size 5.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/5 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 10 | Total steps = 20
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,4.007217
2,3.408619
3,3.967716
4,3.113941
5,3.012876
6,1.791461
7,2.078480
8,2.439763
9,1.934285
10,1.300178


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-20/tokenizer_config.json.


In [8]:
# Set model to evaluation mode for quick testing
FastLanguageModel.for_inference(model)

test_prompt = """Summarize this executive paragraph and explain the technical/financial terms inside it:
'Our development engineering group successfully transitioned all monolithic apps into distributed microservices. To optimize computing infrastructure, we deployed these workloads inside Docker containers managed by local Kubernetes node pools. This structural pivot directly stabilized our corporate balance sheet by shrinking server-side compute expenses by 40%.'"""

inputs = tokenizer(
[
    f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n{test_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
], return_tensors = "pt").to("cuda")

# Generate response
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
decoded_output = tokenizer.batch_decode(outputs)[0]

# Clean up and print the output beautifully
clean_response = decoded_output.split("<|start_header_id|>assistant<|end_header_id|>\n")[-1].replace("<|eot_id|>", "")
print(clean_response)


Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

Summary: The executive paragraph describes the successful migration of legacy monolithic applications to microservices architecture, leveraging Docker containers and Kubernetes orchestration, resulting in a significant reduction of server-side compute expenses.

Technical/Financial Terms Breakdown:

1. **Monolithic apps**: Traditional software architecture where multiple functionalities are tightly coupled into a single, large-scale application.
2. **Microservices**: An architectural approach dividing applications into smaller, independent services communicating via APIs, allowing for scalability, flexibility, and faster development.
3. **Docker containers**: Lightweight, portable, and isolated environments for deploying applications, ensuring consistent deployment across environments.
4. **Kubernetes**: A container orchestration platform automating deployment, scaling, and management of containerized applications.
5. **Local Kubernetes node pools**: A group of Docker containers runnin

In [ ]:
# Convert and compress our custom weights into standard 4-bit GGUF format
model.save_pretrained_gguf("industry_expert_model", tokenizer, quantization_method = "q4_k_m")

# Trigger automatic browser download from Colab's virtual drive
from google.colab import files
files.download("industry_expert_model-unsloth.Q4_K_M.gguf")


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in industry_expert_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]